# Skills (progressive disclosure) en LangChain

Laboratorio sobre *skills* de LangChain (basado en la guía oficial [Build a SQL assistant with on-demand skills](https://docs.langchain.com/oss/python/langchain/multi-agent/skills-sql-assistant)).

## ¿Qué es una "skill" en este patrón?

Una **skill** es un bloque de conocimiento especializado (instrucciones, esquemas, políticas) que **no** se inyecta completo en el system prompt desde el inicio. En su lugar:

1. Solo la **descripción corta** de cada skill (1-2 líneas) va siempre en el system prompt.
2. El **contenido completo** se carga solo cuando el modelo decide llamar una tool (`load_skill`) porque lo necesita.

Esto se llama **progressive disclosure** ("revelación progresiva"):

- **Reduce el uso de contexto**: solo se cargan 1-2 skills relevantes por conversación, no todas.
- **Escala mejor**: se pueden tener decenas de skills sin saturar el prompt.
- **Simplifica el mantenimiento**: cada skill se puede editar de forma independiente.

## Caso de uso de este laboratorio

Usaremos un ejemplo simplificado (subset) del caso de uso real integrado en el API: un agente de una academia que ofrece **rutas de aprendizaje por área** (ej. *Desarrollo de Software* y *Data Science & IA*), con prerrequisitos y progresión de nivel. Cada área es una skill.

!pip install -q -U langchain langchain-openai langchain-core langgraph python-dotenv

## Setup

Cargamos el `.env` (a la misma altura que este notebook, `02_agent_api/notebooks/.env`) y la API key de OpenAI, igual que en `create_agent_demo.ipynb`.

In [1]:
import getpass
import os

from dotenv import load_dotenv

load_dotenv(".env")

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Ingresa tu OPENAI_API_KEY: ")

OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")

## 1. Definir las skills

Cada skill es un `TypedDict` con tres campos:

- `name`: identificador único (lo usa el modelo para llamar `load_skill`).
- `description`: 1-2 frases que **siempre** estarán visibles en el system prompt.
- `content`: la guía completa y detallada, que solo se carga cuando se necesita.

Definimos dos skills de ejemplo (subset del caso de uso real de la academia): `desarrollo_software` y `data_science_ia`.

In [2]:
from typing import List, TypedDict


class Skill(TypedDict):
    """Una skill que se puede revelar progresivamente al agente (progressive disclosure)."""

    name: str  # Identificador único de la skill
    description: str  # 1-2 frases mostradas siempre en el system prompt
    content: str  # Contenido completo con instrucciones detalladas


SKILLS: List[Skill] = [
    {
        "name": "desarrollo_software",
        "description": (
            "Ruta de aprendizaje, prerrequisitos y progresión de nivel para el "
            "área de Desarrollo de Software (backend, frontend y full-stack)."
        ),
        "content": (
            "# Ruta de aprendizaje: Desarrollo de Software\n\n"
            "## Niveles\n"
            "1. **Básico**: lógica de programación, fundamentos de un lenguaje "
            "(Python o JavaScript), control de versiones con Git.\n"
            "2. **Intermedio**: estructuras de datos, POO, bases de datos "
            "relacionales (SQL), APIs REST.\n"
            "3. **Avanzado**: arquitectura de software, testing automatizado, "
            "contenedores (Docker), CI/CD.\n\n"
            "## Prerrequisitos\n"
            "- Intermedio requiere aprobar el módulo Básico.\n"
            "- Avanzado requiere un proyecto final aprobado del nivel Intermedio.\n\n"
            "## Recomendación de progresión\n"
            "No saltar niveles: cada nivel construye las bases del siguiente."
        ),
    },
    {
        "name": "data_science_ia",
        "description": (
            "Ruta de aprendizaje, prerrequisitos y progresión de nivel para el "
            "área de Data Science e Inteligencia Artificial."
        ),
        "content": (
            "# Ruta de aprendizaje: Data Science & IA\n\n"
            "## Niveles\n"
            "1. **Básico**: estadística y probabilidad, Python para análisis "
            "de datos (pandas, numpy).\n"
            "2. **Intermedio**: machine learning supervisado y no supervisado, "
            "feature engineering.\n"
            "3. **Avanzado**: deep learning, NLP, IA generativa (LLMs, RAG).\n\n"
            "## Prerrequisitos\n"
            "- Se recomienda álgebra lineal y cálculo básico antes de Intermedio.\n"
            "- Avanzado requiere aprobar machine learning supervisado.\n\n"
            "## Recomendación de progresión\n"
            "Reforzar estadística y Python antes de avanzar a modelos complejos."
        ),
    },
]

SKILLS

[{'name': 'desarrollo_software',
  'description': 'Ruta de aprendizaje, prerrequisitos y progresión de nivel para el área de Desarrollo de Software (backend, frontend y full-stack).',
  'content': '# Ruta de aprendizaje: Desarrollo de Software\n\n## Niveles\n1. **Básico**: lógica de programación, fundamentos de un lenguaje (Python o JavaScript), control de versiones con Git.\n2. **Intermedio**: estructuras de datos, POO, bases de datos relacionales (SQL), APIs REST.\n3. **Avanzado**: arquitectura de software, testing automatizado, contenedores (Docker), CI/CD.\n\n## Prerrequisitos\n- Intermedio requiere aprobar el módulo Básico.\n- Avanzado requiere un proyecto final aprobado del nivel Intermedio.\n\n## Recomendación de progresión\nNo saltar niveles: cada nivel construye las bases del siguiente.'},
 {'name': 'data_science_ia',
  'description': 'Ruta de aprendizaje, prerrequisitos y progresión de nivel para el área de Data Science e Inteligencia Artificial.',
  'content': '# Ruta de apr

## 2. Tool `load_skill`

Esta tool es el mecanismo por el cual el modelo carga el contenido completo de una skill **solo cuando lo necesita**. El *docstring* es la descripción que ve el modelo para decidir cuándo llamarla.

In [3]:
from langchain.tools import tool


@tool
def load_skill(skill_name: str) -> str:
    """Carga el contenido completo de una skill de orientación curricular.

    Úsala cuando el usuario pida una ruta de aprendizaje, prerrequisitos o
    progresión de nivel dentro de un área académica específica.

    Args:
        skill_name: nombre de la skill a cargar (ej. "desarrollo_software",
            "data_science_ia").
    """
    for skill in SKILLS:
        if skill["name"] == skill_name:
            return f"Skill cargada: {skill_name}\n\n{skill['content']}"

    available = ", ".join(s["name"] for s in SKILLS)
    return f"Skill '{skill_name}' no encontrada. Skills disponibles: {available}"

## 3. `SkillMiddleware`

El middleware es responsable de dos cosas:

1. En `__init__`, construye la lista de **descripciones** de todas las skills (no el contenido completo).
2. En `wrap_model_call`, agrega esas descripciones al final del `system_message` en cada llamada al modelo, y registra `load_skill` como tool disponible vía el atributo de clase `tools`.

Así, el modelo siempre ve *qué* skills existen (nombre + descripción corta), pero solo carga el contenido completo de las que realmente necesita.

In [4]:
from typing import Callable

from langchain.agents.middleware import AgentMiddleware, ModelRequest, ModelResponse
from langchain.messages import SystemMessage


class SkillMiddleware(AgentMiddleware):
    """Middleware que inyecta las descripciones de las skills en el system prompt
    y expone la tool load_skill para cargarlas on-demand (progressive disclosure)."""

    tools = [load_skill]

    def __init__(self) -> None:
        skills_list = [f"- **{skill['name']}**: {skill['description']}" for skill in SKILLS]
        self.skills_prompt = "\n".join(skills_list)

    def wrap_model_call(
        self,
        request: ModelRequest,
        handler: Callable[[ModelRequest], ModelResponse],
    ) -> ModelResponse:
        skills_addendum = (
            f"\n\n## Skills disponibles\n\n{self.skills_prompt}\n\n"
            "Usa la herramienta load_skill cuando necesites la guía detallada "
            "de una de estas áreas académicas."
        )
        new_content = list(request.system_message.content_blocks) + [
            {"type": "text", "text": skills_addendum}
        ]
        new_system_message = SystemMessage(content=new_content)
        modified_request = request.override(system_message=new_system_message)
        return handler(modified_request)

## 4. Crear el agente con soporte de skills

Igual que en `AgentService.__init__` del API, creamos el agente con `create_agent`, pero aquí **sin** `retrieve_documents` ni `get_weather`, para enfocarnos solo en skills. El middleware se pasa en la lista `middleware`.

In [5]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model=OPENAI_MODEL)

agent = create_agent(
    model=llm,
    system_prompt=(
        "Eres un asistente de orientación académica de una academia. "
        "Responde siempre en español y sé conciso."
    ),
    middleware=[SkillMiddleware()],
)

## 5. Probar progressive disclosure

Le hacemos una pregunta que requiere la guía de una skill específica. Observa la secuencia de mensajes: `Human → AI (tool_call load_skill) → Tool (contenido de la skill) → AI (respuesta final)`.

In [6]:
from langchain_core.messages import HumanMessage

response = agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="¿Qué ruta de aprendizaje me recomiendas para entrar a Data Science e IA y cuáles son los prerrequisitos?"
            )
        ]
    }
)

for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

¿Qué ruta de aprendizaje me recomiendas para entrar a Data Science e IA y cuáles son los prerrequisitos?
================================== Ai Message ==================================
Tool Calls:
  load_skill (call_yl9qIRSucz7bT4dnzvzGsSPc)
 Call ID: call_yl9qIRSucz7bT4dnzvzGsSPc
  Args:
    skill_name: data_science_ia
================================= Tool Message =================================
Name: load_skill

Skill cargada: data_science_ia

# Ruta de aprendizaje: Data Science & IA

## Niveles
1. **Básico**: estadística y probabilidad, Python para análisis de datos (pandas, numpy).
2. **Intermedio**: machine learning supervisado y no supervisado, feature engineering.
3. **Avanzado**: deep learning, NLP, IA generativa (LLMs, RAG).

## Prerrequisitos
- Se recomienda álgebra lineal y cálculo básico antes de Intermedio.
- Avanzado requiere aprobar machine learning supervisado.

## Recomendación de pro

También podemos probar con una pregunta que no requiere ninguna skill (respuesta directa, sin `tool_call`), y una que dispara la otra skill (`desarrollo_software`).

In [7]:
response_generic = agent.invoke(
    {"messages": [HumanMessage(content="Hola, ¿qué puedes hacer?")]}
)
for message in response_generic["messages"]:
    message.pretty_print()

print("\n" + "=" * 80 + "\n")

response_dev = agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="Quiero entrar a Desarrollo de Software, ¿qué necesito saber antes de empezar el nivel avanzado?"
            )
        ]
    }
)
for message in response_dev["messages"]:
    message.pretty_print()

================================ Human Message =================================

Hola, ¿qué puedes hacer?
================================== Ai Message ==================================

Hola, puedo ayudarte proporcionando orientación académica en áreas como Desarrollo de Software y Data Science e Inteligencia Artificial. Puedo ofrecerte rutas de aprendizaje, prerrequisitos y progresiones de nivel en estas áreas. ¿En qué te gustaría que te ayude hoy?


================================ Human Message =================================

Quiero entrar a Desarrollo de Software, ¿qué necesito saber antes de empezar el nivel avanzado?
================================== Ai Message ==================================
Tool Calls:
  load_skill (call_rkCD6GiKfFwL0vOryTVqhz2L)
 Call ID: call_rkCD6GiKfFwL0vOryTVqhz2L
  Args:
    skill_name: desarrollo_software
================================= Tool Message =================================
Name: load_skill

Skill cargada: desarrollo_software

# Ruta

## 6. Comparar el ahorro de contexto

Comparamos cuántos caracteres tendría el system prompt si se inyectara el **contenido completo** de todas las skills desde el inicio, contra el enfoque de **progressive disclosure** (solo descripciones + carga on-demand).

In [8]:
descriptions_only = "\n".join(f"- **{s['name']}**: {s['description']}" for s in SKILLS)
full_content = "\n\n".join(s["content"] for s in SKILLS)

print(f"Caracteres solo con descripciones (siempre en el prompt): {len(descriptions_only)}")
print(f"Caracteres si se inyectara TODO el contenido de las skills: {len(descriptions_only) + len(full_content)}")
print(f"Ahorro aproximado: {len(full_content)} caracteres no cargados salvo que se necesiten")

Caracteres solo con descripciones (siempre en el prompt): 294
Caracteres si se inyectara TODO el contenido de las skills: 1431
Ahorro aproximado: 1137 caracteres no cargados salvo que se necesiten


## Conexión con el API

Esta misma implementación (`Skill`, `SKILLS`, `load_skill`, `SkillMiddleware`) está integrada en `src/services/agent_service.py` del proyecto `02_agent_api`, con 4 skills reales (`desarrollo_software`, `data_science_ia`, `diseno_ux_ui`, `ciberseguridad`) y sumada como `middleware=[SkillMiddleware()]` al agente que ya usa `retrieve_documents` (RAG vectorial) y `get_weather`. El `SYSTEM_PROMPT` del servicio se ajustó para indicar al modelo cuándo debe usar `load_skill` en lugar de `retrieve_documents`.